# Stock Analyst Audit Report

Repository: `https://github.com/kinola-IQ/stock-analyst`  
Branch: `main`  
Mode: read-only source inspection

## Executive summary

The repository is a FastAPI application that orchestrates a Google ADK runner, an Ollama-backed agent, yfinance market-data retrieval, and auxiliary logging/storage utilities. The visible tree includes `main.py`, `Interface/`, `system/`, `tests/`, `.github/`, `Dockerfile`, `docker-entrypoint.sh`, and `requirements.txt`. The codebase is Python-centric with FastAPI, Pydantic, Uvicorn, GitHub Actions, Docker, and notebook assets. The documented launch path conflicts with the current import structure, and the container build trusts a remote shell installer.

## Prioritized findings

| ID | Severity | Title | Effort (h) |
|---|---:|---|---:|
| F-01 | Critical | Top-level app cannot be started as documented because main.py uses relative imports | 2 |
| F-02 | High | Docker build executes a remote shell script from ollama.com as root | 4 |
| F-03 | Medium | Request path saves streamed events using a non-serializable key and can throw on malformed events | 2 |
| F-04 | Medium | Input validator raises the wrong exception type and mutates model state in a before-validator | 3 |
| F-05 | Medium | Financial data fetches are synchronous and uncached on the request path | 8 |
| F-06 | Low | Dependabot configuration is a placeholder and will not update any ecosystem | 1 |
| F-07 | Medium | CI workflow pushes mutable latest tags from main without provenance controls | 4 |

## High-confidence findings

### F-01
`main.py` uses relative imports while the README tells users to run `uvicorn main:app`. That combination fails under normal Uvicorn startup because `main` is imported as a top-level module. Evidence: `main.py` lines 383-385 and 413; README lines 280-282.

### F-02
The Dockerfile runs `curl -fsSL https://ollama.com/install.sh | sh` in the runtime stage. That is an unpinned remote installer executed during the build. Evidence: Dockerfile lines 403-418.

### F-03
`Interface/routes.py` stores `ResultStorage.save({event.content: event.content.parts[0].text})`, which is fragile and likely invalid as a dictionary key path. Evidence: lines 476-485.

### F-04
`system/utility/schema.py` mutates the model instance in a before-validator and raises `RequestValidationError` directly from a Pydantic validator. Evidence: lines 318-353.

### F-05
`system/agents/finance_agent/tools_config/ticker_tools.py` calls synchronous yfinance APIs on the request path and does not cache results. Evidence: file body and route loop in `Interface/routes.py`.

## Architecture

```mermaid
flowchart TD
    A[main.py / FastAPI app] --> B[Interface.routes]
    B --> C[google.adk Runner]
    C --> D[system.agents.finance_agent.root_agent]
    D --> E[research_agent / google_search]
    D --> F[analyse_ticker]
    F --> G[yfinance]
    F --> H[pandas]
    F --> I[vaderSentiment]
    D --> J[Ollama / llama_index]
    B --> K[ResultStorage]
    B --> L[logger middleware]
    A --> M[InMemorySessionService]

```

## Dependency picture

Direct dependencies from `requirements.txt`: `fastapi`, `uvicorn[standard]`, `yfinance`, `pandas`, `pydantic`, `nest_asyncio`, `vaderSentiment`, `pinecone-client`, `google-genai`, `llama_index`, `ollama`, `pytest`, `pytest-mock`, `pytest-asyncio`, `pytest-benchmark`. Transitive dependencies were not enumerated because the environment could not install or resolve packages.

## Tests and scans

No local clone was possible because DNS resolution to GitHub failed in the container. As a result, linting, static analysis, dependency scans, secret scans, coverage, and container scans were documented but not executed.

## 30/60/90-day roadmap

**30 days**
- Fix the startup/import mismatch.
- Remove the pipe-to-shell Ollama installation.
- Repair the result-storage path and schema validation path.
- Complete Dependabot configuration.

**60 days**
- Add caching and isolate blocking market-data calls.
- Add startup, auth, and route tests.
- Add dependency pinning and a lockfile.

**90 days**
- Harden CI/CD with provenance and release gating.
- Add coverage thresholds and security scans.
- Add observability around upstream latency, memory growth, and startup time.

## Assumptions and confidence

Assumptions: repository contents on GitHub reflect the visible main branch; no hidden files materially change the conclusions; ADK does not magically neutralize blocking I/O.  
Confidence: Critical findings high; security findings high; performance findings medium; transitive dependency scan low; overall medium-high.
